# Day 1 — RAG in One Hour

---

Today, in about 75 minutes, you'll build a **working RAG system** end-to-end.

**RAG** stands for **Retrieval-Augmented Generation**. In one sentence:

> Look up the right documents first, then hand them to the LLM along with the question.

That's the whole idea. Every "chat with your PDF", "AI support bot", or "enterprise knowledge assistant" is a RAG system.


## 1. Why RAG exists

LLMs have two big weaknesses:

1. **Their knowledge is frozen.** GPT-4 doesn't know what your company shipped last Tuesday.
2. **They hallucinate.** Ask an LLM about something outside its training data and it will confidently make things up.

**RAG fixes both** by fetching relevant text *at query time* and adding it to the prompt.

```
Without RAG:               With RAG:
  User asks question         User asks question
        │                          │
        ▼                          ▼
      [LLM]                   [Search KB]  ← finds top-k relevant chunks
        │                          │
        ▼                          ▼
     answer                   [Prompt: chunks + question]
   (maybe wrong)                    │
                                    ▼
                                  [LLM]
                                    │
                                    ▼
                                  answer (grounded in your docs)
```


## 2. The three steps

Every RAG system has exactly three steps:

1. **Retrieve** — semantic search finds top-k relevant chunks (Section 5, done!)
2. **Augment** — build a prompt that says "here are the docs, answer the question"
3. **Generate** — send that prompt to an LLM (Section 4, done!)

That's it. The rest of this section is just making each step better.


## 3. Setup — Chroma + Together AI


In [ ]:
!pip install sentence-transformers chromadb together python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Confirm the key is loaded
assert os.getenv("TOGETHER_API_KEY"), "Set TOGETHER_API_KEY in .env"


## 4. Build a mini knowledge base

We'll seed Chroma with 6 tiny docs about a made-up company "AcmeCloud" so the LLM cannot possibly know the answers from its training data.


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
kb = client.create_collection("acme_kb")

docs = [
    "AcmeCloud's free tier includes 10 GB of storage and 100 API calls per day.",
    "The Pro plan costs $29/month and includes 500 GB of storage and unlimited API calls.",
    "AcmeCloud servers are located in AWS us-east-1 and eu-west-1 regions.",
    "To reset your password, click 'Forgot Password' on the login page or email support@acmecloud.io.",
    "Enterprise customers receive 24/7 phone support and a dedicated account manager.",
    "AcmeCloud was founded in 2019 by Priya Rao and Marcus Chen in Austin, Texas.",
]

kb.add(
    documents=docs,
    embeddings=model.encode(docs).tolist(),
    ids=[f"doc_{i}" for i in range(len(docs))],
)
print(f"KB ready — {kb.count()} docs")


## 5. Step 1 — Retrieve


In [ ]:
def retrieve(question: str, top_k: int = 3):
    q_vec = model.encode([question]).tolist()
    r = kb.query(query_embeddings=q_vec, n_results=top_k)
    return r["documents"][0]

chunks = retrieve("How much does the Pro plan cost?")
for c in chunks:
    print(" -", c)


## 6. Step 2 — Augment (build the RAG prompt)

The classic RAG prompt has three parts:

1. **System instruction** — "answer using only the context below"
2. **Context** — the retrieved chunks
3. **User question**


In [ ]:
def build_prompt(question: str, chunks: list[str]) -> str:
    context = "\n\n".join(f"- {c}" for c in chunks)
    return f"""You are a helpful assistant for AcmeCloud.
Answer the user's question using ONLY the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

prompt = build_prompt("How much does the Pro plan cost?", chunks)
print(prompt)


## 7. Step 3 — Generate


In [ ]:
from together import Together

llm = Together()

def generate(prompt: str) -> str:
    resp = llm.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content

answer = generate(prompt)
print(answer)


**Expected:** the LLM tells you the Pro plan costs $29/month — a fact it could not possibly know from training. That's RAG working.


## 8. Put it all together — one function


In [ ]:
def rag(question: str, top_k: int = 3) -> str:
    chunks = retrieve(question, top_k)
    prompt = build_prompt(question, chunks)
    return generate(prompt)

for q in [
    "Who founded AcmeCloud?",
    "How do I reset my password?",
    "What is the airspeed velocity of an unladen swallow?",  # not in KB
]:
    print(f"Q: {q}")
    print(f"A: {rag(q)}\n")


**Notice the third question.** The context doesn't contain anything about swallows, so the LLM should say "I don't know" instead of making something up. That's the `only use the context` instruction doing its job.

If your LLM still hallucinates — Day 5 covers how to make it stop.


## 9. Show the sources (bare-bones citations)

Users trust answers more when you show where they came from. A quick fix — return the sources alongside the answer.


In [ ]:
def rag_with_sources(question: str, top_k: int = 3) -> dict:
    chunks = retrieve(question, top_k)
    prompt = build_prompt(question, chunks)
    return {
        "answer": generate(prompt),
        "sources": chunks,
    }

result = rag_with_sources("Where are AcmeCloud servers located?")
print("Answer:", result["answer"])
print("\nSources used:")
for s in result["sources"]:
    print(" -", s)


## Recap

- **RAG = Retrieve → Augment → Generate.** Three steps, that's it.
- The **retrieve** step comes from Section 5. The **generate** step comes from Section 4. Today we glued them together with a **prompt template**.
- Always instruct the LLM to answer *only from the context* — otherwise it hallucinates.
- Show sources — trust and debuggability come for free.
- **Next class:** loading real documents (DOCX, web pages, markdown) into the knowledge base.
